# Pseudohuérfanas: construcción y recuperación del blanco

Construcción de las **k = 1** (drogas con exactamente un blanco proteico conocido
en una especie), su estratificación en grupos 0–3, el `frank` por especie y la
clasificación de la semilla en *nula / no informativa / informativa*.

Cada droga se **orfaniza**: se le sacan todas sus aristas de bioactividad y se
arma la semilla sólo con su vecindario químico. El blanco verdadero puede volver
por otra droga vecina —eso es inferencia directa y es legítima—, nunca por sí
misma (`comun/tests/test_fuga.py`).

Los parámetros son los óptimos por especie de `genome_prioritization/02`, y la
capa química va **filtrada por promiscuidad** (README §7.3): estos números no son
comparables con los de `gon3/resultados/pseudohuerfanas/`.

Fuente: `tdr-graph/orphan_drugs_v4.ipynb`.
Salidas: `01_<especie>_pseudohuerfanas.csv`, `01_k1_drugs.csv`, `01_resumen_frank.csv`.

## Imports

In [ ]:
import sys, os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")          # antes de numpy: un hilo por proceso

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

from pathlib import Path
# raiz del repositorio: se busca hacia arriba la carpeta que tiene DB/,
# asi el notebook corre desde donde sea que se haya clonado
RAIZ = Path.cwd()
while not (RAIZ / "DB").is_dir() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "comun"))
sys.path.insert(0, str(RAIZ / "huerfanas"))
%load_ext autoreload
%autoreload 2
import tdr, nucleo as nf
import funciones_huerfanas as fh              # el .py de esta carpeta

SALIDAS = tdr.out("huerfanas")
FIGURAS = SALIDAS / "figuras"
NB      = "01"                             # numero de este notebook: prefija todo lo que genere
n_core  = 20

plt = tdr.estilo()
SALIDAS

## Datos

In [ ]:
# cluster_consistent=True: `huerfanas/` filtra los positivos/negativos
# inconsistentes a nivel cluster, como orphan_drugs_v4.ipynb.
crudo = tdr.cargar_db(anotaciones=True, quimica=True, cluster_consistent=True)

# README §7.3: el filtro de promiscuidad se APLICA antes de armar ninguna semilla.
# La lista sale de analiceDB/03; si falta, el error dice qué correr.
promiscuos = tdr.compuestos_promiscuos()
datos = tdr.filtrar_capa_quimica(crudo, promiscuos)

optimos = fh.parametros_optimos()          # de genome_prioritization_out/02_*
len(datos.posdt), len(optimos)

## Acondicionamiento

In [ ]:
k1, posdt_sp = fh.construir_pseudohuerfanas(datos)
especies = [sp for sp in k1["sp_id"].value_counts().index if sp in optimos]
muestras = {sp: fh.muestra_pseudohuerfanas(k1[k1["sp_id"] == sp], n=fh.N_MUESTRA)
            for sp in especies}

k1.to_csv(SALIDAS / f"{NB}_k1_drugs.csv", index=False)
print(f"{len(especies)} especies con pseudohuérfanas y parámetros óptimos")
fig = fh.fig_grupos(k1, plt)
tdr.guardar(fig, f"{NB}_f01_grupos_por_especie", FIGURAS)

## Corrida

In [ ]:
# celda de corrida: el ciclo se lee aca
import time
for sp in especies:
    t0  = time.time()
    ctx = fh.preparar_especie(datos, sp, optimos[sp], k1)         # RS con sus optimos
    fh.fijar_contexto(datos, posdt_sp, ctx)                       # visible por fork
    filas = fh.paralelizar(fh.priorizar_droga, muestras[sp]["drug_id"], n_core=n_core, desc=sp)
    fh.guardar_huerfanas(filas, SALIDAS, NB, sp)                  # -> 01_<sp>_pseudohuerfanas.csv
    print(f"  {sp} en {time.time() - t0:.0f} s", flush=True)

fh.escribir_meta(SALIDAS, NB, notebook="01_pseudohuerfanas.ipynb",
                 params={"optimos": optimos, "n_muestra": fh.N_MUESTRA, "rseed": fh.RSEED},
                 n_core=n_core, especies=especies,
                 filtro_promiscuidad=True, n_promiscuos=len(promiscuos))

# Resultados

In [ ]:
res = fh.cargar_resultados(SALIDAS, NB, patron="*_pseudohuerfanas")
res = res.merge(k1[["drug_id", "sp_id", "grupo"]].rename(columns={"sp_id": "especie"}),
                on=["drug_id", "especie"], how="left")
resumen = fh.resumen_frank(res)
resumen.to_csv(SALIDAS / f"{NB}_resumen_frank.csv", index=False)
resumen[resumen["particion"].isin(["total", "semilla", "grupo"])].round(1)

In [ ]:
fig = fh.fig_frank(res, plt)
tdr.guardar(fig, f"{NB}_f02_frank_por_grupo", FIGURAS)

In [ ]:
# El cuello de botella: la clase de semilla, no la reponderación
fig = fh.fig_cobertura(res, plt)
tdr.guardar(fig, f"{NB}_f03_semilla_por_especie", FIGURAS)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5), tight_layout=True)
r = resumen[resumen["particion"] == "semilla"].sort_values("pct_recuperadas")
ax.barh(range(len(r)), r["pct_recuperadas"],
        color=[tdr.COLOR_SEMILLA.get(v, tdr.MUTED) for v in r["valor"]])
ax.set_yticks(range(len(r))); ax.set_yticklabels(r["valor"], fontsize=8)
ax.set_xlabel("% con frank < 0.1")
ax.set_title("La recuperación la decide la clase de semilla")
tdr.guardar(fig, f"{NB}_f04_recuperacion_por_semilla", FIGURAS)